# Rabi AWG marker/reference test (AWG only, no other instruments)

Purpose: before building the full Rabi sweep, verify on a scope whether the `DATA:SEQ` sequence table's per-segment `marker_mode` field (e.g. `"highAtStart"`) fires **once when a `"repeat"`-type segment's whole N-times-repeated block starts**, or **once every individual repeat** (N times). This codebase has only ever used `marker_mode` on `"once"`-type segments (see `t1_test.py`'s `readout` segment, `pulsed_odmr.py`'s CH1 `readout` segment) -- never on a `"repeat"`-type segment -- so this specific behavior is unverified.

This matters because the planned Rabi lock-in reference works like this: CH1 plays the *same* laser-pulse-plus-gap "rep" arb for two back-to-back blocks of `N_REPS` repeats each, with the first block's sequence-table entry marked `"highAtStart"` and the second marked `"lowAtStart"`. If the marker only toggles once per block, CH1's Sync BNC output is a clean slow square wave -- exactly what the SR830's external reference needs. If it instead fires on every repeat, the Sync output would be a burst of N quick edges during the "on" block followed by silence during the "off" block, which is *not* a valid periodic reference for the lock-in to phase-lock to.

This notebook drives **only the AWG** (CH1 = laser pulse train + block marker, CH2 = MW gate pulse, same physical setup as `pulsed_odmr.py`) -- no generator, lock-in, PSUs, or interlock. Nothing here reads anything back electronically; you verify by probing with a real oscilloscope:

1. **CH1 analog output**: should show `N_REPS` bright laser pulses (each followed by a dark gap containing where the MW pulse would go), then `N_REPS` more identical pulses, repeating forever -- same shape both blocks, since CH1's *analog* signal doesn't change between blocks, only the marker does.
2. **CH2 analog output**: should show a real MW gate pulse inside the dark gap for the first `N_REPS` reps ("mw-on" block), then flat low for the next `N_REPS` reps ("mw-off" block).
3. **CH1's Sync/marker BNC output** (this is the one that matters): put it on the scope and check whether it's a single clean square wave at the block period, or a train of pulses. Report back what you see -- if it's multiple edges instead of one clean transition per block, the sequence-table marker approach doesn't work as planned and we'll need the fallback (a single big pre-concatenated arb per block with per-sample marker data, if that's even supported -- separate thing to check).
4. Optionally, scope CH1 and CH2 together to eyeball whether `PHASe:SYNChronize` is actually keeping them aligned (same open item flagged in `notes.md`'s pulsed-ODMR section for `pulsed_odmr.py`).

## Connect to the AWG

In [25]:
import pyvisa
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('USB0::0x0957::0x5707::MY53800810::INSTR', 'USB0::0xF4EC::0x1103::SDG1XDDX6R5043::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR')


In [54]:
import sys
sys.path.insert(0, "..")

import numpy as np
import ks33600a

AWG_RESOURCE = "USB0::0x0957::0x5707::MY53800810::INSTR"

awg = ks33600a.KS33600A(AWG_RESOURCE, debug=True)

Keysight 33600A: connected
*RST => +0,"No error"
*CLS => +0,"No error"
SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"


## Parameters

Kept small/fast so the block period is easy to find on a scope. `N_REPS` is exactly the thing we're testing the marker behavior against -- start small (so you can visually count edges) and bump it up once you've confirmed the basic behavior.

`MW_US` is a fixed representative pulse width here, NOT swept -- this notebook only tests the marker/reference mechanism, not the real Rabi sweep.

In [62]:
FS = 1e9  # sample rate, same convention as t1_test.py / pulsed_odmr.py

LASER_US = 2.0   # laser pulse duration
PRE_US = 1.0     # padding before the MW pulse (settle time)
MW_US = 2.0      # MW pulse duration (tau_mw) -- fixed here, swept in the real experiment
POST_US = 1.0    # padding after the MW pulse before the next laser pulse --
                  # keeps RF and the next readout from overlapping in time,
                  # same rationale as the CW-ODMR RF-pickup fix (see notes.md)

N_REPS = 50  # reps per block -- start small, increase once marker behavior is confirmed

REP_US = LASER_US + PRE_US + MW_US + POST_US
BLOCK_US = N_REPS * REP_US
print(f"one rep = {REP_US} us, one block = {BLOCK_US} us, "
      f"full reference period = {2 * BLOCK_US} us ({1 / (2 * BLOCK_US * 1e-6):.1f} Hz)")

one rep = 6.0 us, one block = 300.0 us, full reference period = 600.0 us (1666.7 Hz)


## Build the arb waveforms

CH1's `rep` arb is used, UNCHANGED, for both blocks -- only the sequence table's `marker_mode` differs between the two entries that reference it, which is exactly the thing under test.

CH2 needs two different arbs, since its actual gate signal (not just the marker) differs between blocks: `gate_on_rep` (low, then a real MW pulse, then low) for the mw-on block, and `gate_off_rep` (low the whole time) for the mw-off block. Both must have the exact same total sample count as CH1's `rep` arb, or the two channels' blocks would drift out of step -- checked with an assert below (same lesson as `pulsed_odmr.py`'s `us_to_samples()` rounding-consistency comment).

### Clear volatile memory before (re-)uploading

`DATA:ARBitrary`/`upload_waveform()` errors if an arb name already exists in volatile memory (confirmed in the manual, page 241) -- and `upload_waveform()` doesn't check `SYST:ERR?` after the binary transfer the way `write()` does (see `ks33600a.py`), so that error would be silently swallowed. This is a real gap in the driver, confirmed by reading the code and the manual. **Not independently confirmed, though, that this was the actual cause of any specific symptom seen in this notebook** (e.g. the `pre_us` gap issue) -- that was a plausible hypothesis at the time, not a verified diagnosis; the gap was never re-checked after adding this cell before other issues took priority. Keep running this cell before `build-waveforms` as a precaution when re-uploading without reconnecting, but don't treat its presence as proof any earlier bug is actually fixed.

In [63]:
awg.write("SOUR1:DATA:VOL:CLE")
awg.write("SOUR2:DATA:VOL:CLE")
print("Volatile waveform memory cleared on both channels.")

SOUR1:DATA:VOL:CLE => +0,"No error"
SOUR2:DATA:VOL:CLE => +0,"No error"
Volatile waveform memory cleared on both channels.


In [64]:
def us_to_samples(duration_us):
    return max(1, round(duration_us * 1e-6 * FS))


def rf_pulse(freq_hz, n_samples):
    t = np.arange(n_samples) / FS
    return np.sin(2 * np.pi * freq_hz * t).astype(np.float32)


def const(n_samples, value):
    return np.full(n_samples, value, dtype=np.float32)


laser_samples = us_to_samples(LASER_US)
pre_samples = us_to_samples(PRE_US)
mw_samples = us_to_samples(MW_US)
post_samples = us_to_samples(POST_US)

# CH1: one bright laser pulse followed by a flat dark gap spanning where
# the MW pulse goes on CH2 -- CH1 itself doesn't care about the MW timing
# internal structure, only the total gap length.
ch1_rep = np.concatenate([
    rf_pulse(80e6, laser_samples),
    const(pre_samples + mw_samples + post_samples, 0.0),
])

# CH2, mw-on block: low during the laser pulse + pre-padding, high during
# the MW pulse, low during post-padding. +1/-1 normalized, mapped to a real
# 0-5V swing in the output-config cell below (same convention as
# pulsed_odmr.py's gate_pre/gate_high/gate_post).
ch2_gate_on_rep = np.concatenate([
    const(laser_samples + pre_samples, -1.0),
    const(mw_samples, 1.0),
    const(post_samples, -1.0),
])

# CH2, mw-off block: low the entire rep -- switch parked on the dump path
# throughout, no MW pulse at all.
ch2_gate_off_rep = const(laser_samples + pre_samples + mw_samples + post_samples, -1.0)

assert len(ch1_rep) == len(ch2_gate_on_rep) == len(ch2_gate_off_rep), (
    "CH1 and CH2 rep arbs must have identical sample counts, or the two "
    "channels' blocks will drift out of step"
)
print(f"rep length: {len(ch1_rep)} samples ({len(ch1_rep) / FS * 1e6:.3f} us)")

# Anchor segment for the "start a sequence on a trigger" technique (manual,
# p.181): a brief DC waveform played once ("onceWaitTrig"), then the
# sequence holds and waits for a trigger before advancing into the real
# content. Minimum segment length for 33600 Series is 32 Sa.
#
# IMPORTANT: each channel's anchor must sit at that channel's own real
# "off"/rest level, not a generic 0.0 -- confirmed on real hardware that
# reusing a plain 0.0-valued anchor on CH2 produced a brief, unwanted
# ~2.5V blip (the midpoint of CH2's 0-5V unipolar VOLT/VOLT:OFFS mapping,
# where 0.0 normalized is NOT the same physical level as the "off" state,
# unlike CH1's simple 0-centered convention where 0.0 normalized already
# is the correct rest level) right after the MW-on block ends and before
# the real gate_off_rep segment pulls it down to the true 0V off level.
ANCHOR_SAMPLES = 32
anchor_ch1 = const(ANCHOR_SAMPLES, 0.0)   # matches ch1_rep's own "off" level
anchor_ch2 = const(ANCHOR_SAMPLES, -1.0)  # matches gate_off_rep's low level (-> 0V)

print(f"anchor length: {len(anchor_ch1)} samples ({len(anchor_ch1) / FS * 1e6:.4f} us)")

awg.upload_waveform(ch1_rep, arb_name="rep", ch=1, sample_rate=FS)
awg.upload_waveform(ch2_gate_on_rep, arb_name="gate_on_rep", ch=2, sample_rate=FS)
awg.upload_waveform(ch2_gate_off_rep, arb_name="gate_off_rep", ch=2, sample_rate=FS)
awg.upload_waveform(anchor_ch1, arb_name="anchor", ch=1, sample_rate=FS)
awg.upload_waveform(anchor_ch2, arb_name="anchor", ch=2, sample_rate=FS)

rep length: 6000 samples (6.000 us)
anchor length: 32 samples (0.0320 us)
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 1
FORM:BORD SWAP => +0,"No error"
Keysight 33600A: uploaded arb to channel 2


## Build and upload the sequences

**Now using the manual's documented "start a sequence on a trigger" technique (p.181, page-quoted verbatim by the user): place a brief DC waveform in front of the other segments, marked `onceWaitTrig`, so the sequence plays it once then holds and waits for a trigger before advancing.** This is a *sequence-level* trigger (segment-advance-on-trigger), distinct from the burst/sweep-only `TRIGger:SOURce` restriction found earlier -- the manual's own "Waveform Sequencing Applications" example explicitly describes using "an external hardware trigger" to advance between sequence segments, so this should genuinely respect `TRIGger[1|2]:SOURce`. Both channels get an `anchor` segment (`onceWaitTrig`) prepended; once both have played their anchor and are sitting in the wait-for-trigger state, a single shared trigger event should advance both into their real sequences at the same instant -- a fundamentally different (and hopefully more reliable) mechanism than the failed `PHASe:SYNChronize`/`TRIG:SOUR BUS`+`INIT`+`*TRG`/gated-burst attempts.

Marker moved to CH2's own sequence (still): `highAtStart` on the real mw-on block, `lowAtStart` on mw-off, with the shared Sync/Marker BNC sourced from CH2 (`OUTPut:SYNC:SOURce CH2` in the output-config cell). This was already working regardless of CH1/CH2 timing -- kept as-is; the anchor/trigger mechanism here is specifically to also fix CH1-vs-CH2 relative timing, which the marker relocation alone didn't address.

CH2 is targeted via the `SOUR2:DATA:SEQ` prefix, same as `pulsed_odmr.py` used -- confirmed accepted without error on this firmware.

In [65]:
def build_block_descriptor(sequence_name, segments):
    """Same DATA:SEQ block-descriptor builder as t1_test.py/pulsed_odmr.py."""
    parts = [f'"{sequence_name}"']
    for arb_name, repeat_count, play_control, marker_mode, marker_point in segments:
        parts.append(
            f'"{arb_name}",{repeat_count},{play_control},{marker_mode},{marker_point}'
        )
    payload = ",".join(parts)
    payload_bytes = payload.encode("utf-8")
    payload_len = len(payload_bytes)
    n = len(str(payload_len))
    return f"#{n}{payload_len}{payload}"


SEQUENCE_NAME_CH1 = "rabi_marker_test_ch1_v4"
SEQUENCE_NAME_CH2 = "rabi_marker_test_ch2_v4"

# CH1: "anchor" segment first, played once then waiting for a trigger
# before advancing (manual p.181's documented technique) -- marker doesn't
# matter here (moved to CH2), "maintain" throughout.
block1_segments = [
    ["anchor", "1", "onceWaitTrig", "maintain", 10],
    ["rep", str(N_REPS), "repeat", "maintain", 10],
    ["rep", str(N_REPS), "repeat", "maintain", 10],
]
block1 = build_block_descriptor(SEQUENCE_NAME_CH1, block1_segments)
awg.write(f"DATA:SEQ {block1}")  # unprefixed -> channel 1, per t1_test.py's convention

# CH2: same anchor/wait-for-trigger structure, then the real marker-bearing
# content (highAtStart on the real mw-on block, lowAtStart on mw-off).
block2_segments = [
    ["anchor", "1", "onceWaitTrig", "lowAtStart", 10],
    ["gate_off_rep", str(N_REPS), "repeat", "lowAtStart", 10],
    ["gate_on_rep", str(N_REPS), "repeat", "highAtStart", 10],
]
block2 = build_block_descriptor(SEQUENCE_NAME_CH2, block2_segments)
awg.write(f"SOUR2:DATA:SEQ {block2}")

DATA:SEQ #3117"rabi_marker_test_ch1_v4","anchor",1,onceWaitTrig,maintain,10,"rep",50,repeat,maintain,10,"rep",50,repeat,maintain,10 => +0,"No error"
SOUR2:DATA:SEQ #3141"rabi_marker_test_ch2_v4","anchor",1,onceWaitTrig,lowAtStart,10,"gate_off_rep",50,repeat,lowAtStart,10,"gate_on_rep",50,repeat,highAtStart,10 => +0,"No error"


## Configure channel output and start

Same output configuration as `t1_test.py`/`pulsed_odmr.py`: CH1 into 50 ohm at a small Vpp (laser drive convention), CH2 into a high-impedance load with a 0-5V swing (ZYSWA switch control levels). Amplitude is set with plain `VOLTage`/`VOLTage:OFFSet` -- confirmed on real hardware that `FUNC:ARB:PTPeak`/`FUNC:ARB:PTP` does NOT actually update the channel's real output amplitude register on this firmware.

**Using `TRIG:SOUR EXT` with the `onceWaitTrig` anchor segment (previous cell), triggered by a real external edge from the SDG1062X.** `TRIG:SOUR BUS` + `*TRG` produced no output at all -- not yet clear whether that's because `*TRG` doesn't work for sequence-advance triggering the way it does for burst/sweep, or something else. Testing with a genuine external hardware edge next, both to try to get real output and to isolate whether `BUS`/`*TRG` specifically was the problem. `OUTPut:SYNC:SOURce CH2` still switches the marker source to CH2.

In [66]:
# CH1: laser/AOM drive -- waits at the "anchor" segment for a trigger.
awg.write("OUTP1:LOAD 50")
awg.write("SOUR1:FUNC:ARB:SRAT 1e9")
awg.write(f'SOUR1:FUNC:ARB "{SEQUENCE_NAME_CH1}"')
awg.write("SOUR1:FUNC ARB")
awg.write("SOUR1:VOLT 0.632")
awg.write("TRIG1:SOUR EXT")
awg.write("TRIG1:SLOP POS")
awg.write("TRIG1:LEV 1.5")
awg.write("OUTPUT1 ON")

# CH2: MW gate -> ZYSWA switch control -- same idea.
awg.write("OUTP2:LOAD INF")
awg.write("SOUR2:FUNC:ARB:SRAT 1e9")
awg.write(f'SOUR2:FUNC:ARB "{SEQUENCE_NAME_CH2}"')
awg.write("SOUR2:FUNC ARB")
awg.write("SOUR2:VOLT 5.0")
awg.write("SOUR2:VOLT:OFFS 2.5")
awg.write("TRIG2:SOUR EXT")
awg.write("TRIG2:SLOP POS")
awg.write("TRIG2:LEV 1.5")
awg.write("OUTPUT2 ON")

# Switch the single shared Sync/Marker BNC to source from CH2 instead of
# the default CH1 -- ties the reference signal to CH2's own real MW-gate
# output (see sequences-header markdown above).
awg.write("OUTPut:SYNC:SOURce CH2")

print("Both channels waiting at their 'anchor' segment for an external "
      "trigger edge on the rear-panel Ext Trig BNC. Run the SDG1062X cell "
      "below to supply it.")

OUTP1:LOAD 50 => +0,"No error"
SOUR1:FUNC:ARB:SRAT 1e9 => +0,"No error"
SOUR1:FUNC:ARB "rabi_marker_test_ch1_v4" => +0,"No error"
SOUR1:FUNC ARB => +0,"No error"
SOUR1:VOLT 0.632 => +0,"No error"
TRIG1:SOUR EXT => +0,"No error"
TRIG1:SLOP POS => +0,"No error"
TRIG1:LEV 1.5 => +0,"No error"
OUTPUT1 ON => +0,"No error"
OUTP2:LOAD INF => +0,"No error"
SOUR2:FUNC:ARB:SRAT 1e9 => +0,"No error"
SOUR2:FUNC:ARB "rabi_marker_test_ch2_v4" => +0,"No error"
SOUR2:FUNC ARB => +0,"No error"
SOUR2:VOLT 5.0 => +0,"No error"
SOUR2:VOLT:OFFS 2.5 => +0,"No error"
TRIG2:SOUR EXT => +0,"No error"
TRIG2:SLOP POS => +0,"No error"
TRIG2:LEV 1.5 => +0,"No error"
OUTPUT2 ON => +0,"No error"
OUTPut:SYNC:SOURce CH2 => +0,"No error"
Both channels waiting at their 'anchor' segment for an external trigger edge on the rear-panel Ext Trig BNC. Run the SDG1062X cell below to supply it.


## Supply the external trigger edge with the SDG1062X

Configures the Siglent SDG1062X (a separate function generator on the bench, driven by `sdg1062x.py`) as a plain continuous square-wave source -- NOT using its own `SDG1062X.run()` method, which configures it to be externally *triggered itself* (the opposite of what's needed here). Wire its CH1 output to the Keysight's rear-panel **Ext Trig** BNC.

**Important, confirmed on real hardware: this needs to be a CONTINUOUS, fast trigger, not a one-time edge.** When the sequence finishes its last segment (`gate_on_rep`) it wraps back to the FIRST segment to loop -- which is the `anchor` -- and since that's `onceWaitTrig`, every single lap needs a fresh trigger edge. With a slow trigger (100 Hz, tried first) the sequence blazed through one off+on cycle right after each edge, then sat frozen at the anchor (holding low) for most of the 10 ms until the next edge -- an extremely asymmetric duty cycle instead of the intended 50/50 square wave. The fix is to drive the external trigger at (or above) the real block-cycle rate (`1 / (2 * BLOCK_US)`, printed in the `params` cell) so every wraparound is retriggered essentially immediately. For the real experiment this means the external trigger frequency needs to be reconfigured per sweep point, since `BLOCK_US` depends on `MW_US`, which is swept -- a real design implication, not just a test-notebook detail.

In [67]:
import sdg1062x

SDG_RESOURCE = "USB0::0xF4EC::0x1103::SDG1XDDX6R5043::INSTR"

sdg = sdg1062x.SDG1062X(SDG_RESOURCE, debug=True)

# Must be >= 1 / (2 * BLOCK_US) (see params cell) so every sequence
# wraparound gets retriggered before it would otherwise sit idle at the
# anchor. 50 kHz got the duty cycle much closer to 50/50 but not exact --
# expected, since this trigger free-runs asynchronously to the sequence's
# own wrap-around moment, so each lap picks up a random extra dead time of
# up to one trigger period (~20us at 50kHz) waiting for the next edge,
# which always lengthens the "low" (anchor) portion. Going much faster
# (1 MHz, ~1us period) bounds that worst-case extra dead time to ~1.7% of
# the 60us block cycle instead of ~33% -- won't ever be exactly 50/50 with
# an unsynchronized trigger, but should be close enough not to matter.
TRIGGER_FREQ_HZ = 1_000_000

sdg.write("C1:BSWV WVTP,SQUARE")
sdg.write(f"C1:BSWV FRQ,{TRIGGER_FREQ_HZ}")
sdg.write("C1:BSWV AMP,5")     # 5 Vpp
sdg.write("C1:BSWV OFST,2.5")  # 0-5V swing, comfortably above the 1.5V
                                 # threshold set on the Keysight above
sdg.write("C1:OUTP ON")

print(f"SDG1062X CH1 outputting a {TRIGGER_FREQ_HZ} Hz, 0-5V square wave -- "
      "wire this into the Keysight's rear-panel Ext Trig BNC.")

*RST
Siglent SDG1062X: connected
C1:BSWV WVTP,SQUARE
C1:BSWV FRQ,1000000
C1:BSWV AMP,5
C1:BSWV OFST,2.5
C1:OUTP ON
SDG1062X CH1 outputting a 1000000 Hz, 0-5V square wave -- wire this into the Keysight's rear-panel Ext Trig BNC.


In [32]:
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR2:FUNC?", awg.query("SOUR2:FUNC?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("SOUR2:VOLT?", awg.query("SOUR2:VOLT?"))
print("SOUR2:VOLT:OFFS?", awg.query("SOUR2:VOLT:OFFS?"))
print("SOUR2:DATA:VOL:CAT?", awg.query("SOUR2:DATA:VOL:CAT?"))

OUTP2? 1
SOUR2:FUNC? ARB
SOUR2:FUNC:ARB? "rabi_marker_test_ch2_v4"
SOUR2:VOLT? +5.0000000000000E+00
SOUR2:VOLT:OFFS? +2.5000000000000E+00
SOUR2:DATA:VOL:CAT? "EXP_RISE","GATE_ON_REP","GATE_OFF_REP","ANCHOR","rabi_marker_test_ch2_v4"


## What to check on the oscilloscope

- **CH1 analog out**: `N_REPS` bright pulses, repeating continuously after the initial anchor/trigger.
- **CH2 analog out**: a real gate pulse inside the dark gap for `N_REPS` reps, then flat low for the next `N_REPS` reps, repeating.
- **CH1 vs CH2 relative timing (the main thing this anchor/trigger mechanism is supposed to fix)**: does CH1's laser pulse land in a consistent place relative to CH2's MW pulse? Try re-running just the `configure-output` cell (which re-selects the sequence, re-triggers via `*TRG`, and restarts both channels from their anchors) a few times and see if the relationship is the same each time, or still varies.
- **The Sync/marker BNC** (sourced from CH2): should still be a clean square wave at the block period, high during CH2's real MW-pulse block. Should be unaffected by the anchor/trigger addition, but worth re-confirming.

## Diagnostic: confirm the instrument's actual state

Useful any time the scope doesn't show what's expected -- queries the AWG directly rather than trusting that the writes above "must have" taken effect. If `OUTP1?`/`OUTP2?` come back `0`, the outputs never actually turned on. If they come back `1` and everything else below looks right but the scope still shows nothing, check the sequence structure itself -- confirmed on this hardware that a `DATA:SEQ` sequence made entirely of `"repeat"`-type segments (no `"once"` segment) reports a fully healthy state here yet produces no real output; see the note above `build-sequences` for the fix already applied.

In [14]:
print("OUTP1?", awg.query("OUTP1?"))
print("OUTP2?", awg.query("OUTP2?"))
print("SOUR1:FUNC?", awg.query("SOUR1:FUNC?"))
print("SOUR2:FUNC?", awg.query("SOUR2:FUNC?"))
print("SOUR1:FUNC:ARB?", awg.query("SOUR1:FUNC:ARB?"))
print("SOUR2:FUNC:ARB?", awg.query("SOUR2:FUNC:ARB?"))
print("TRIG1:SOUR?", awg.query("TRIG1:SOUR?"))
print("TRIG2:SOUR?", awg.query("TRIG2:SOUR?"))
print("SYST:ERR?", awg.query("SYST:ERR?"))  # anything still sitting in the
                                              # error queue that debug=True
                                              # printed but didn't stop on

OUTP1? 1
OUTP2? 1
SOUR1:FUNC? ARB
SOUR2:FUNC? ARB
SOUR1:FUNC:ARB? "rabi_marker_test_ch1"
SOUR2:FUNC:ARB? "rabi_marker_test_ch2"
TRIG1:SOUR? IMM
TRIG2:SOUR? IMM
SYST:ERR? +0,"No error"


## Stop / disconnect

Run when done probing.

In [ ]:
awg.write("OUTPUT1 OFF")
awg.write("OUTPUT2 OFF")
awg.close()
sdg.write("C1:OUTP OFF")
sdg.close()
print("AWG and SDG1062X outputs off, connections closed.")